# Computer Vision: Image Representation, Spatial Filtering & Modern Augmentations

This interactive lab covers:
1. **Image Representations**: Channel-wise normalization (ImageNet statistics).
2. **Spatial Convolutions & Filtering**: 2D Gaussian smoothing and Sobel gradient edge detection.
3. **Regularization through Augmentations**: Cutout, Mixup, and CutMix data augmentation pipelines.

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('04-computer-vision/image-processing/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))

import numpy as np
import torch
from transforms_filters import (
    normalize_image,
    gaussian_kernel,
    apply_conv2d,
    sobel_edge_detector,
    cutout,
    mixup,
    cutmix,
)

print('Image Processing & Augmentation module loaded successfully!')

## 1. Image Representation & Normalization

Standardizing RGB image tensors using ImageNet channel mean and standard deviations.

In [ ]:
# Create a synthetic RGB image (H=32, W=32, C=3)
np.random.seed(42)
raw_image = np.random.randint(0, 256, (32, 32, 3), dtype=np.uint8)
normalized = normalize_image(raw_image)

print(f'Raw image range:        [{raw_image.min()}, {raw_image.max()}]')
print(f'Normalized mean per ch:  {normalized.mean(axis=(0, 1))}')
print(f'Normalized std per ch:   {normalized.std(axis=(0, 1))}')

## 2. Spatial Convolutions & Edge Detection (Sobel & Gaussian)

Applying 2D spatial convolution filters for Gaussian blur and directional Sobel edge gradients.

In [ ]:
# Generate a 5x5 Gaussian kernel
g_kernel = gaussian_kernel(size=5, sigma=1.2)
blurred = apply_conv2d(raw_image.astype(np.float32) / 255.0, g_kernel)

# Sobel edge detection on blurred image
gx, gy, magnitude = sobel_edge_detector(blurred)

print(f'Gaussian kernel sum:       {g_kernel.sum():.4f}')
print(f'Sobel Gx range:            [{gx.min():.3f}, {gx.max():.3f}]')
print(f'Gradient magnitude mean:   {magnitude.mean():.4f}')

## 3. Advanced Data Augmentations: Cutout, Mixup, and CutMix

Implementing modern regularization strategies that prevent over-fitting in deep CNNs and Vision Transformers.

In [ ]:
# 1. Cutout
cutout_img = cutout(raw_image, mask_size=10)
erased_pixels = (cutout_img == 0).all(axis=-1).sum()
print(f'Cutout erased {erased_pixels} pixels')

# 2. Mixup
torch.manual_seed(42)
t_img1 = torch.rand((3, 32, 32))
t_img2 = torch.rand((3, 32, 32))
y1 = torch.tensor([1.0, 0.0])  # Class 0
y2 = torch.tensor([0.0, 1.0])  # Class 1

mixed_x, mixed_y = mixup(t_img1, t_img2, y1, y2, alpha=0.4)
print(f'Mixup blended target labels: {mixed_y.numpy()}')

# 3. CutMix
cm_x, cm_y = cutmix(t_img1, t_img2, y1, y2, alpha=1.0)
print(f'CutMix blended target labels: {cm_y.numpy()}')